# Bronze Layer - Data Ingestion

Ingests raw chocolate sales data from Kaggle and loads into Delta tables.

**Source:** Kaggle dataset (ssssws/chocolate-sales-dataset-2023-2024)  
**Target:** workspace.bronze_chocolate schema  
**Format:** Delta tables with audit columns

In [0]:
%pip install kagglehub --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import kagglehub
import os
from glob import glob
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

BRONZE_CATALOG = "workspace"
BRONZE_SCHEMA = "bronze_chocolate"

✓ Imports loaded
✓ Target: workspace.bronze_chocolate


In [0]:
path = kagglehub.dataset_download("ssssws/chocolate-sales-dataset-2023-2024")
csv_files = glob(os.path.join(path, "*.csv"))

print(f"Downloaded {len(csv_files)} files to {path}")

📦 Downloading chocolate sales dataset from Kaggle...
✓ Dataset downloaded to: /home/spark-e10676b6-5770-4697-907c-6f/.cache/kagglehub/datasets/ssssws/chocolate-sales-dataset-2023-2024/versions/2

✓ Found 5 CSV file(s):
  • calendar.csv (19.2 KB)
  • products.csv (9.3 KB)
  • customers.csv (1513.2 KB)
  • stores.csv (4.5 KB)
  • sales.csv (69265.8 KB)


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA}")

✓ Schema workspace.bronze_chocolate ready


In [0]:
bronze_tables = {}

for csv_file in csv_files:
    file_name = os.path.basename(csv_file)
    table_name = file_name.replace('.csv', '').replace('-', '_').replace(' ', '_').lower()
    
    pdf = pd.read_csv(csv_file)
    df = spark.createDataFrame(pdf)
    
    df_bronze = df \
        .withColumn("bronze_ingestion_timestamp", current_timestamp()) \
        .withColumn("bronze_source_file", lit(file_name))
    
    table_path = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_{table_name}"
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_path)
    
    bronze_tables[table_name] = table_path
    print(f"Loaded {df_bronze.count():,} rows -> {table_path}")


📊 Processing: calendar.csv -> bronze_calendar
  • Rows: 731
  • Columns: 6
  • Rows: 731
  • Columns: 8
  • Schema: date, year, month, day, week...
  ✓ Saved to: workspace.bronze_chocolate.bronze_calendar

📊 Processing: products.csv -> bronze_products
  • Rows: 200
  • Columns: 6
  • Rows: 200
  • Columns: 8
  • Schema: product_id, product_name, brand, category, cocoa_percent...
  ✓ Saved to: workspace.bronze_chocolate.bronze_products

📊 Processing: customers.csv -> bronze_customers
  • Rows: 50,000
  • Columns: 5
  • Rows: 50,000
  • Columns: 7
  • Schema: customer_id, age, gender, loyalty_member, join_date...
  ✓ Saved to: workspace.bronze_chocolate.bronze_customers

📊 Processing: stores.csv -> bronze_stores
  • Rows: 100
  • Columns: 5
  • Rows: 100
  • Columns: 7
  • Schema: store_id, store_name, city, country, store_type...
  ✓ Saved to: workspace.bronze_chocolate.bronze_stores

📊 Processing: sales.csv -> bronze_sales
  • Rows: 1,000,000
  • Columns: 11
  • Rows: 1,000,000
  • Co

In [0]:
print("\nBronze layer summary:")
for table_name, table_path in bronze_tables.items():
    row_count = spark.table(table_path).count()
    print(f"{table_path}: {row_count:,} rows")


BRONZE LAYER TABLES
  • workspace.bronze_chocolate.bronze_calendar: 731 rows
  • workspace.bronze_chocolate.bronze_products: 200 rows
  • workspace.bronze_chocolate.bronze_customers: 50,000 rows
  • workspace.bronze_chocolate.bronze_stores: 100 rows
  • workspace.bronze_chocolate.bronze_sales: 1,000,000 rows

✓ Bronze tables are ready for Silver layer processing!
